# 🔹 Condução, Convecção e Radiação Térmica
## Volume II – Capítulo 3 | Nível: Graduação/Pós-Graduação

**Objetivos de aprendizagem:**
- Aplicar a Lei de Fourier e calcular resistências térmicas
- Estimar coeficientes convectivos por correlações
- Calcular troca radiativa pela lei de Stefan-Boltzmann
- Resolver problemas acoplados de condução-convecção-radiação

**Referência:** Lugon Junior (2026), Vol II, Cap. 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

SIGMA = 5.67e-8  # W/(m²·K⁴)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 🔸 Resistências Térmicas em Série

In [ ]:
def R_conducao(k, L, A):
    """Resistência térmica de condução para parede plana."""
    return L / (k * A)

def R_conveccao(h, A):
    """Resistência térmica de convecção."""
    return 1 / (h * A)

def R_rad_linearizada(eps, T_m, A):
    """Resistência radiativa linearizada em torno de T_m."""
    h_rad = 4 * eps * SIGMA * T_m**3
    return 1 / (h_rad * A)

# Exercício Vol II, Cap 3, Ex 1: Parede composta
A = 10.0  # m²

# Camadas
camadas = [
    {'k': 0.7, 'L': 0.15, 'nome': 'Tijolo'},
    {'k': 0.025, 'L': 0.05, 'nome': 'Poliuretano'},
    {'k': 0.5, 'L': 0.02, 'nome': 'Gesso'}
]

R_total = 0
print("Resistências térmicas (parede composta):")
for cam in camadas:
    R = R_conducao(cam['k'], cam['L'], A)
    R_total += R
    print(f"  {cam['nome']:12s}: R = {R:.4f} K/W")

print(f"\nResistência TOTAL: R_total = {R_total:.4f} K/W")

# Fluxo para ΔT = 20 K
dT = 20  # K
q = dT / R_total
print(f"Fluxo de calor para ΔT = {dT} K: q = {q:.2f} W")

## 🔸 Troca Radiativa: Forma Não-Linear vs Linearizada

In [ ]:
def q_rad_nao_linear(eps, T_s, T_inf):
    """Fluxo radiativo líquido (forma exata)."""
    return eps * SIGMA * (T_s**4 - T_inf**4)

def q_rad_linearizada(eps, T_s, T_inf):
    """Fluxo radiativo linearizado em torno de T_m."""
    T_m = (T_s + T_inf) / 2
    h_rad = 4 * eps * SIGMA * T_m**3
    return h_rad * (T_s - T_inf)

# Exercício Vol II, Cap 3, Ex 3
eps = 0.90
T_s = 40 + 273.15   # 40°C → K
T_inf = -10 + 273.15 # -10°C → K

q_nl = q_rad_nao_linear(eps, T_s, T_inf)
q_lin = q_rad_linearizada(eps, T_s, T_inf)
erro = abs(q_nl - q_lin) / q_nl * 100

print(f"\nRadiação (ε={eps}, T_s={T_s-273.15:.0f}°C, T_∞={T_inf-273.15:.0f}°C):")
print(f"  Forma não-linear:  q'' = {q_nl:.1f} W/m²")
print(f"  Forma linearizada: q'' = {q_lin:.1f} W/m²")
print(f"  Erro da linearização: {erro:.2f}%")

# Varredura: erro vs ΔT
dT_vals = np.linspace(10, 100, 20)
erros = []
T_base = 298  # 25°C

for dT in dT_vals:
    T_s_var = T_base + dT/2
    T_inf_var = T_base - dT/2
    q_nl = q_rad_nao_linear(eps, T_s_var, T_inf_var)
    q_lin = q_rad_linearized(eps, T_s_var, T_inf_var)
    erros.append(abs(q_nl - q_lin) / q_nl * 100)

plt.figure(figsize=(8, 5))
plt.plot(dT_vals, erros, 'b-o', markersize=4)
plt.xlabel('Diferença de temperatura ΔT [K]')
plt.ylabel('Erro da linearização [%]')
plt.title('Validade da linearização radiativa (ε = 0.90)')
plt.grid(True, alpha=0.3)
plt.axhline(y=5, color='r', linestyle='--', label='Limite 5%')
plt.legend()
plt.tight_layout()
plt.show()

## 🔸 Caso de Estudo: Isolamento de Forno (Problema Acoplado)

In [ ]:
def dimensionar_isolamento(T_int, T_amb, h_int, h_ext, eps, k_iso, L_aco, A, T_ext_max):
    """
    Dimensiona espessura de isolante para limitar T_ext.
    
    Retorna:
    --------
    L_iso_min : float
        Espessura mínima do isolante [m]
    """
    # Resistências fixas
    R_conv_int = R_conveccao(h_int, A)
    R_aco = R_conducao(45, L_aco, A) * 2  # duas chapas de aço
    
    # Linearizar radiação em torno de T_ext_max
    T_m = (T_ext_max + T_amb) / 2
    h_rad = 4 * eps * SIGMA * T_m**3
    h_total_ext = h_ext + h_rad
    R_conv_ext = R_conveccao(h_total_ext, A)
    
    # Resistência total alvo para atingir T_ext_max
    q_alvo = h_total_ext * A * (T_ext_max - T_amb)
    R_total_alvo = (T_int - T_amb) / q_alvo
    
    # Resistência do isolante necessária
    R_iso = R_total_alvo - (R_conv_int + R_aco + R_conv_ext)
    
    # Espessura mínima
    L_iso_min = R_iso * k_iso * A
    
    return L_iso_min, q_alvo, h_total_ext

# Parâmetros do forno (Caso de Estudo Vol II, Cap 3)
T_int = 250 + 273.15   # 250°C → K
T_amb = 25 + 273.15    # 25°C → K
h_int = 15             # W/(m²·K)
h_ext = 10             # W/(m²·K)
eps = 0.90
k_iso = 0.04           # W/(m·K) - lã de vidro
L_aco = 0.003          # 3 mm
A = 2.0                # m² por face
T_ext_max = 60 + 273.15 # 60°C → K

L_iso, q, h_ext_tot = dimensionar_isolamento(
    T_int, T_amb, h_int, h_ext, eps, k_iso, L_aco, A, T_ext_max
)

print(f"\nDimensionamento de isolamento de forno:")
print(f"  Espessura mínima do isolante: L_iso = {L_iso*1000:.1f} mm")
print(f"  Fluxo de calor: q = {q:.1f} W")
print(f"  Coeficiente externo total: h_ext,total = {h_ext_tot:.2f} W/(m²·K)")
print(f"\nRecomendação prática: Adotar L_iso = {np.ceil(L_iso*1000/5)*5:.0f} mm (margem de segurança)")

## ✅ Exercícios Propostos

1. **[Graduação]** Calcule a perda térmica por metro de tubo de aço (k=45 W/m·K, D_i=50 mm, D_e=60 mm) com vapor a 150°C, ar externo a 25°C, h_o=15 W/m²·K.
2. **[Pós-Graduação]** Implemente esquema implícito para condução 1D com condição de contorno radiativa não-linear e teste convergência com refinamento de malha.

> 💡 Dica: Para problemas não-lineares, use iteração de ponto fixo ou Newton-Raphson na condição de contorno.